In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings

warnings.filterwarnings('ignore') # Para ignorar warnings de scikit-learn

# =============================================================================
# 1. CARGA DE LOS DATOS ÓPTIMOS (Ganadores del Wrapper RandomForest)
# =============================================================================
# Usamos los archivos que ya están escalados y recortados a las 10 mejores variables
train_df = pd.read_csv('train_selected_RandomForest.csv')
tuning_df = pd.read_csv('tuning_selected_RandomForest.csv')

target_col = 'Class'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_tuning = tuning_df.drop(columns=[target_col])
y_tuning = tuning_df[target_col]

# Aseguramos que la clase sea numérica de forma segura
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_tuning_num = le.transform(y_tuning)

# =============================================================================
# 2. INICIALIZACIÓN DE LOS 4 MODELOS BASE (Shallow Learning)
# =============================================================================
# Se instancian con su configuración por defecto (base). 
# probability=True en SVM es necesario para poder calcular la métrica ROC-AUC.
modelos = {
    'Regresión Logística': LogisticRegression(random_state=42),
    'SVM (Lineal)': SVC(kernel='linear', probability=True, random_state=42),
    'k-NN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# =============================================================================
# 3. ENTRENAMIENTO Y EVALUACIÓN
# =============================================================================
resultados = []

print("Entrenando y evaluando modelos bases sobre el conjunto de Tuning...\n")

for nombre, modelo in modelos.items():
    # Entrenamiento exclusivo con el conjunto de Train
    modelo.fit(X_train, y_train_num)
    
    # Predicción sobre el conjunto de Tuning (Validación)
    y_pred = modelo.predict(X_tuning)
    y_prob = modelo.predict_proba(X_tuning)
    
    # Manejo dinámico para clasificación binaria o multiclase
    is_multiclass = len(le.classes_) > 2
    avg_method = 'macro' if is_multiclass else 'binary'
    
    # Cálculo de métricas
    acc = accuracy_score(y_tuning_num, y_pred)
    prec = precision_score(y_tuning_num, y_pred, average=avg_method)
    rec = recall_score(y_tuning_num, y_pred, average=avg_method)
    f1 = f1_score(y_tuning_num, y_pred, average=avg_method)
    
    # ROC-AUC (Para multiclase se usa ovo, para binario solo la probabilidad de la clase positiva)
    if is_multiclass:
        roc = roc_auc_score(y_tuning_num, y_prob, multi_class='ovo')
    else:
        roc = roc_auc_score(y_tuning_num, y_prob[:, 1])
        
    resultados.append({
        'Modelo': nombre,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc
    })

# =============================================================================
# 4. REPORTE DE COMPARACIÓN
# =============================================================================
df_resultados = pd.DataFrame(resultados).set_index('Modelo')

print("="*80)
print( "COMPARACIÓN DE MODELOS BASE (Métricas sobre Tuning Set)")
print("="*80)
print(df_resultados.round(4).to_string())
print("="*80)

Entrenando y evaluando modelos bases sobre el conjunto de Tuning...

COMPARACIÓN DE MODELOS BASE (Métricas sobre Tuning Set)
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Modelo                                                             
Regresión Logística    0.7581     0.7353  0.8065    0.7692   0.9022
SVM (Lineal)           0.7903     0.7500  0.8710    0.8060   0.9043
k-NN                   0.8387     0.8182  0.8710    0.8438   0.8746
Random Forest          0.8548     0.8667  0.8387    0.8525   0.9412
